In [ ]:
# 1. Prepare sentiment data for training
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Load sentiment data
news_df = pd.read_csv("news_with_sentiment.csv")
news_df['published_date'] = pd.to_datetime(news_df['published_date']).dt.date

# Aggregate daily sentiment by stock
daily_sentiment = news_df.groupby(['stock_symbol', 'published_date']).agg({
    'sentiment': lambda x: x.value_counts().index[0],
    'confidence': 'mean',
    'text_length': 'mean'
}).reset_index()

# Encode sentiment: positive=2, neutral=1, negative=0
le = LabelEncoder()
daily_sentiment['sentiment_encoded'] = le.fit_transform(daily_sentiment['sentiment'])

# Create sequences for each stock
def create_sentiment_sequences(data, seq_length=10):
    X, dates = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i])
        dates.append(data.index[i])
    return np.array(X), dates

# Train LSTM for each stock
models = {}
for ticker in ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN']:
    stock_sentiment = daily_sentiment[daily_sentiment['stock_symbol'] == ticker]
    stock_sentiment = stock_sentiment.sort_values('published_date')
    
    features = stock_sentiment[['sentiment_encoded', 'confidence']].values
    X, prediction_dates = create_sentiment_sequences(features)
    
    # Create dummy targets (we'll predict movement direction)
    y = stock_sentiment['sentiment_encoded'].iloc[10:].values
    
    model = Sequential([
        LSTM(50, input_shape=(X.shape[1], X.shape[2])),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(X, y > 1, epochs=50, verbose=0)  # Predict positive sentiment
    models[ticker] = model

print("✅ Sentiment models trained")